# April 05
Reorganized the Reward Signals in utils.py; now have 6 different signals.
Training this new reward shaping from scratch using train_ray_selfplay.py.

Submitted 2 training jobs: one CPU version, one GPU version.
Waiting for results; if GPU works fine, will use GPU-only training going forward.

NEXT:
Once results are in, run a match against ceia_baseline_agent to evaluate performance.

In [ ]:
#SBATCH --job-name=GPU_S2_selfplay
#SBATCH -N1 --ntasks-per-node=12 --gres=gpu:V100:1
#SBATCH --mem-per-cpu=6G
#SBATCH -t12:00:00
#SBATCH --output=soccerstwos-%j.out
#SBATCH --mail-type=END,FAIL
#SBATCH --mail-user=frank.yang@gatech.edu


source ~/miniconda3/etc/profile.d/conda.sh
conda activate soccertwos

# Fix protobuf version conflict with ray==1.4.0
export PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION=python
pip install "numpy==1.23.5" -q  # fix numpy version conflict with ray==1.4.0

# Run the training script
/storage/ice1/7/4/fyang365/.conda/conda_envs/soccertwos/bin/python train_ray_selfplay.py


# April 06
In this project, GPU only handles the PPO model update (forward/backward pass). The soccer-twos environment is a Unity process; the simulation itself cannot be GPU-accelerated, and the MLP is too small for GPU to provide any meaningful gain.
Use CPU-only training from here on.


### Current Script Does Not Support Multi-Node
train_ray_selfplay.py:40
ray.init()  # only initialises local Ray, unaware of other nodes
Multi-node requires changing to `ray.init(address='auto')` and manually starting a Ray cluster (head + workers) in the Slurm script — fairly complex.

### Fix: Cannot Resume Training
Problem: type information is lost during checkpoint serialisation.
Chain of events:
Ray saves a checkpoint by first converting the optimizer state dict (which contains torch tensors) to numpy arrays, then serialising with pickle.
Certain tensors (e.g. Adam's step counter, a scalar int64) get stored as `numpy.object_` instead of a concrete numeric type during conversion.
On restore, Ray tries to convert these numpy arrays back to torch tensors by calling `torch.from_numpy()`, but torch refuses to convert `object_` dtype.

Fix:
Go to `/storage/ice1/7/4/fyang365/.conda/conda_envs/soccertwos/lib/python3.8/site-packages/ray/rllib/utils/torch_utils.py`
In `def mapping(item)`, replace
```
# Everything else: Convert to numpy, then wrap as torch tensor.
else:
    tensor = torch.from_numpy(np.asarray(item))
```
with:
```
# Everything else: Convert to numpy, then wrap as torch tensor.
else:
    arr = np.asarray(item)
    if arr.dtype == object:
        arr = np.array(arr.tolist(), dtype=np.float32)
    # Scalar values (like lr) should stay as Python scalars, not tensors
    if arr.ndim == 0:
        return arr.item()
    tensor = torch.from_numpy(arr)
```


# April 07
Re-running with Ray 1.4.

# April 08
Using `python -m soccer_twos.evaluate     -m1 reward_shaping_ppo_agent     -m2 ceia_baseline_agent     -e 100`
Compared the Ray 1.4 reward-shaping version against ceia:
The reward-shaping version beats the TA's original ceia baseline by 9 percentage points.
Need to continue training.

# April 09
Current training results:
- `PPO_Soccer_4923f_00000_0_2026-04-08_21-56-12` — Ray 1.13, checkpoint 1300
- `PPO_Soccer_36ca0_00000_0_2026-04-07_18-00-41` — Ray 1.4, checkpoint 1000
- `PPO_Soccer_1874b_00000_0_2026-04-09_13-18-16` — Ray 1.4 version; full environment recorded in environment.yml


# April 10
`PPO_Soccer_bc696_00000_0_2026-04-09_14-12-58`
- Ray 1.4, eliminated the `DeprecationWarning: wrapping <function policy_mapping_fn at`
- Trained to `scratch/soccer-twos/ray_results/PPO_selfplay_rec/PPO_Soccer_bc696_00000_0_2026-04-09_14-12-58/checkpoint_000700/checkpoint-700`
- Continued training from here: `PPO_Soccer_0b340_00000_0_2026-04-10_15-47-02`

### Comparison Test:
`python -m soccer_twos.watch -m1 reward_shaping_ppo_agent -m2 ceia_baseline_agent`
- The above requires a display — local machine only.
- Use this on the server:
```
python -m soccer_twos.evaluate \
    -m1 reward_shaping_ppo_agent \
    -m2 ceia_baseline_agent \
    -e 100
```


### checkpoint-700 Results
```
Progress: |██████████████████████████████████████████████████| 100 / 100 episodes completed
episode_len_mean: 79.32
episode_reward_max: -0.013599991798400879
episode_reward_mean: -0.15827199816703796
episode_reward_min: -1.6103999614715576
episodes_this_eval: 100
policies:
  ceia_baseline_agent:
    blue_team:
      policy_blue_team_draws: 0
      policy_blue_team_losses: 19
      policy_blue_team_reward_max: 1.981600046157837
      policy_blue_team_reward_mean: 0.39344000816345215
      policy_blue_team_reward_min: -2.0
      policy_blue_team_total_games: 50
      policy_blue_team_win_rate: 0.62
      policy_blue_team_wins: 31
    orange_team:
      policy_orange_team_draws: 0
      policy_orange_team_losses: 27
      policy_orange_team_reward_max: 1.9847999811172485
      policy_orange_team_reward_mean: -0.2342880219221115
      policy_orange_team_reward_min: -2.0
      policy_orange_team_total_games: 50
      policy_orange_team_win_rate: 0.46
      policy_orange_team_wins: 23
    policy_draws: 0
    policy_losses: 46
    policy_reward_max: 1.9847999811172485
    policy_reward_mean: 0.07957600802183151
    policy_reward_min: -2.0
    policy_total_games: 100
    policy_win_rate: 0.54
    policy_wins: 54
  reward_shaping_ppo_agent:
    blue_team:
      policy_blue_team_draws: 0
      policy_blue_team_losses: 23
      policy_blue_team_reward_max: 1.979599952697754
      policy_blue_team_reward_mean: 0.06441599130630493
      policy_blue_team_reward_min: -2.0
      policy_blue_team_total_games: 50
      policy_blue_team_win_rate: 0.54
      policy_blue_team_wins: 27
    orange_team:
      policy_orange_team_draws: 0
      policy_orange_team_losses: 31
      policy_orange_team_reward_max: 1.9864000082015991
      policy_orange_team_reward_mean: -0.5401120185852051
      policy_orange_team_reward_min: -2.0
      policy_orange_team_total_games: 50
      policy_orange_team_win_rate: 0.38
      policy_orange_team_wins: 19
    policy_draws: 0
    policy_losses: 54
    policy_reward_max: 1.9864000082015991
    policy_reward_mean: -0.23784799873828888
    policy_reward_min: -2.0
    policy_total_games: 100
    policy_win_rate: 0.46
    policy_wins: 46
```

# April 11
Obtained `scratch/soccer-twos/ray_results/PPO_selfplay_rec/PPO_Soccer_0b340_00000_0_2026-04-10_15-47-02/checkpoint_002000/checkpoint-2000`.
Match results against ceia_baseline_agent are very poor.

```
Progress: |██████████████████████████████████████████████████| 100 / 100 episodes completed
episode_len_mean: 73.18
episode_reward_max: -0.01639997959136963
episode_reward_mean: -0.1459999978542328
episode_reward_min: -0.6967999935150146
episodes_this_eval: 100
policies:
  ceia_baseline_agent:
    blue_team:
      policy_blue_team_draws: 0
      policy_blue_team_losses: 5
      policy_blue_team_reward_max: 1.9836000204086304
      policy_blue_team_reward_mean: 1.452552080154419
      policy_blue_team_reward_min: -2.0
      policy_blue_team_total_games: 50
      policy_blue_team_win_rate: 0.9
      policy_blue_team_wins: 45
    orange_team:
      policy_orange_team_draws: 0
      policy_orange_team_losses: 10
      policy_orange_team_reward_max: 1.9836000204086304
      policy_orange_team_reward_mean: 1.0916880369186401
      policy_orange_team_reward_min: -2.0
      policy_orange_team_total_games: 50
      policy_orange_team_win_rate: 0.8
      policy_orange_team_wins: 40
    policy_draws: 0
    policy_losses: 15
    policy_reward_max: 1.9836000204086304
    policy_reward_mean: 1.2721199989318848
    policy_reward_min: -2.0
    policy_total_games: 100
    policy_win_rate: 0.85
    policy_wins: 85
  reward_shaping_ppo_agent:
    blue_team:
      policy_blue_team_draws: 0
      policy_blue_team_losses: 40
      policy_blue_team_reward_max: 1.9648000001907349
      policy_blue_team_reward_mean: -1.230159878730774
      policy_blue_team_reward_min: -2.0
      policy_blue_team_total_games: 50
      policy_blue_team_win_rate: 0.2
      policy_blue_team_wins: 10
    orange_team:
      policy_orange_team_draws: 0
      policy_orange_team_losses: 45
      policy_orange_team_reward_max: 1.9783999919891357
      policy_orange_team_reward_mean: -1.6060800552368164
      policy_orange_team_reward_min: -2.0
      policy_orange_team_total_games: 50
      policy_orange_team_win_rate: 0.1
      policy_orange_team_wins: 5
    policy_draws: 0
    policy_losses: 85
    policy_reward_max: 1.9783999919891357
    policy_reward_mean: -1.4181201457977295
    policy_reward_min: -2.0
    policy_total_games: 100
    policy_win_rate: 0.15
    policy_wins: 15
```

### Root Bug: Opponents Never Update
See train_ray_selfplay.py:27:


class SelfPlayUpdateCallback(DefaultCallbacks):
    def on_train_result(self, **info):
        if info["result"]["episode_reward_mean"] > 0.5:  # ← the problem is here
            print("---- Updating opponents!!! ----")
This condition is never True.

episode_reward_mean is the total reward including shaped rewards, from utils.py:35:


combined = {aid: reward[aid] + shaped[aid] for aid in reward}
The shaped reward contains two permanently negative signals:

Signal 5 (per-step time penalty): -0.001 × steps, ~75 steps/episode → -0.075 per episode
Signal 4 (dangerous defence): deducting points whenever the ball is in play
Your actual data confirms this:

April 10: episode_reward_mean = -0.158
April 11: episode_reward_mean = -0.146
Always negative, never exceeds 0.5.

Consequences

opponent_1 / opponent_2 / opponent_3 were never updated throughout training
↓
They remain frozen at checkpoint-700 weights forever
↓
The default policy spent 1300 iterations exclusively countering its own checkpoint-700 version
↓
Learned an exploit strategy against that specific pattern, completely ineffective against ceia_baseline_agent
This explains why performance degraded the more it trained: the agent did not get dumber — it overfit to an opponent that could never evolve.

Secondary issue
[utils.py:27]: The 0.5 threshold for episode_reward_mean was calibrated for raw reward (original soccer-twos signal is ±2). Adding shaped reward makes this threshold completely wrong.


### NEXT：
Changed to `if info["result"]["episode_reward_mean"] > -0.05:`
Resumed training from checkpoint-700:
- `PPO_Soccer_750bc_00000_0_2026-04-11_14-30-04`

Updated training script to:
`"num_workers": 7,
"num_envs_per_worker": 3,`
This reduces Ray inter-process communication overhead (22 processes → 7).
Rollout collection per worker is more continuous, improving batch quality.
Total parallel environments are about the same (22 → 21), throughput roughly unchanged.
- `PPO_Soccer_6cc90_00000_0_2026-04-11_15-27-06`







# April 12

`scratch/soccer-twos/ray_results/PPO_selfplay_rec/PPO_Soccer_750bc_00000_0_2026-04-11_14-30-04/checkpoint_001900/checkpoint-1900` vs baseline results:

```
Progress: |██████████████████████████████████████████████████| 100 / 100 episodes completed
episode_len_mean: 55.85
episode_reward_max: -0.014799952507019043
episode_reward_mean: -0.11124400049448013
episode_reward_min: -0.45280003547668457
episodes_this_eval: 100
policies:
  ceia_baseline_agent:
    blue_team:
      policy_blue_team_draws: 0
      policy_blue_team_losses: 9
      policy_blue_team_reward_max: 1.9747999906539917
      policy_blue_team_reward_mean: 1.1871360540390015
      policy_blue_team_reward_min: -2.0
      policy_blue_team_total_games: 50
      policy_blue_team_win_rate: 0.82
      policy_blue_team_wins: 41
    orange_team:
      policy_orange_team_draws: 0
      policy_orange_team_losses: 11
      policy_orange_team_reward_max: 1.983199954032898
      policy_orange_team_reward_mean: 1.023919939994812
      policy_orange_team_reward_min: -2.0
      policy_orange_team_total_games: 50
      policy_orange_team_win_rate: 0.78
      policy_orange_team_wins: 39
    policy_draws: 0
    policy_losses: 20
    policy_reward_max: 1.983199954032898
    policy_reward_mean: 1.1055281162261963
    policy_reward_min: -2.0
    policy_total_games: 100
    policy_win_rate: 0.8
    policy_wins: 80
  reward_shaping_ppo_agent:
    blue_team:
      policy_blue_team_draws: 0
      policy_blue_team_losses: 39
      policy_blue_team_reward_max: 1.9811999797821045
      policy_blue_team_reward_mean: -1.1410319805145264
      policy_blue_team_reward_min: -2.0
      policy_blue_team_total_games: 50
      policy_blue_team_win_rate: 0.22
      policy_blue_team_wins: 11
    orange_team:
      policy_orange_team_draws: 0
      policy_orange_team_losses: 41
      policy_orange_team_reward_max: 1.985200047492981
      policy_orange_team_reward_mean: -1.2925119400024414
      policy_orange_team_reward_min: -2.0
      policy_orange_team_total_games: 50
      policy_orange_team_win_rate: 0.18
      policy_orange_team_wins: 9
    policy_draws: 0
    policy_losses: 80
    policy_reward_max: 1.985200047492981
    policy_reward_mean: -1.2167720794677734
    policy_reward_min: -2.0
    policy_total_games: 100
    policy_win_rate: 0.2
    policy_wins: 20
```

```
Progress: |██████████████████████████████████████████████████| 100 / 100 episodes completed
episode_len_mean: 75.78
episode_reward_max: -0.018800020217895508
episode_reward_mean: -0.15107598900794983
episode_reward_min: -0.809999942779541
episodes_this_eval: 100
policies:
  ceia_baseline_agent:
    blue_team:
      policy_blue_team_draws: 0
      policy_blue_team_losses: 8
      policy_blue_team_reward_max: 1.979200005531311
      policy_blue_team_reward_mean: 1.2336320877075195
      policy_blue_team_reward_min: -2.0
      policy_blue_team_total_games: 50
      policy_blue_team_win_rate: 0.84
      policy_blue_team_wins: 42
    orange_team:
      policy_orange_team_draws: 0
      policy_orange_team_losses: 7
      policy_orange_team_reward_max: 1.9811999797821045
      policy_orange_team_reward_mean: 1.321287989616394
      policy_orange_team_reward_min: -2.0
      policy_orange_team_total_games: 50
      policy_orange_team_win_rate: 0.86
      policy_orange_team_wins: 43
    policy_draws: 0
    policy_losses: 15
    policy_reward_max: 1.9811999797821045
    policy_reward_mean: 1.277459979057312
    policy_reward_min: -2.0
    policy_total_games: 100
    policy_win_rate: 0.85
    policy_wins: 85
  reward_shaping_ppo_agent:
    blue_team:
      policy_blue_team_draws: 0
      policy_blue_team_losses: 43
      policy_blue_team_reward_max: 1.9775999784469604
      policy_blue_team_reward_mean: -1.456527829170227
      policy_blue_team_reward_min: -2.0
      policy_blue_team_total_games: 50
      policy_blue_team_win_rate: 0.14
      policy_blue_team_wins: 7
    orange_team:
      policy_orange_team_draws: 0
      policy_orange_team_losses: 42
      policy_orange_team_reward_max: 1.9579999446868896
      policy_orange_team_reward_mean: -1.4005439281463623
      policy_orange_team_reward_min: -2.0
      policy_orange_team_total_games: 50
      policy_orange_team_win_rate: 0.16
      policy_orange_team_wins: 8
    policy_draws: 0
    policy_losses: 85
    policy_reward_max: 1.9775999784469604
    policy_reward_mean: -1.428536057472229
    policy_reward_min: -2.0
    policy_total_games: 100
    policy_win_rate: 0.15
    policy_wins: 15
```

#### Problem:
The two rewards differ by 100×. During training the agent optimises shaped reward (15.15); during evaluation scoring uses raw game reward (±2 max). The agent learned to farm shaped reward instead of winning.

Shaped reward magnitude comparison:

Signal	Coefficient	Estimated per-step	60-step episode total
Ball proximity Signal 1	0.01	~0.03	~1.8/agent
Attack direction Signal 3	0.02	~0.1	~6/agent
Kick ball Signal 2	0.05	sporadic	~0.5/agent
4 agents total			~30+
True game signal		only on goal	±1/goal
Shaped reward is 30-60× the game signal; the agent has no chance of learning the "win" objective.

#### Fix:

Change	File	Effect
Divide all shaped reward coefficients by 50	utils.py	Game signal dominates again; training/eval consistent
Reset callback threshold to > 0.3	train_ray_selfplay.py	Aligned with raw reward scale — "only update opponents after winning"
Remove restore	train_ray_selfplay.py	Train from scratch; avoid carrying over bad habits
After retraining from scratch, episode_reward_mean should be in the range -0.5 to +1.5 (close to raw game reward scale) — that is the correct state.

In [ ]:
### Training from scratch
- Reason: adjusted shaped reward coefficients
- New training log: `soccerstwos-4805405.out`
- `PPO_Soccer_eb80a_00000_0_2026-04-12_02-57-50`


In [ ]:
### Iteration 900 Results:
- 33% win rate at iteration 900 from scratch is normal. The reward scale is now correct: rewards: min=-1.0, max=0.84, mean=0.001
- mean ≈ 0.001 (close to 0) means the game signal is dominating; shaped reward is no longer interfering. This is the correct state.

#### However:
- Early iterations (1-3):
- episode_reward_mean: 0.564, 0.913, 1.025  ← callback triggered
- episode_len_mean: 627, 757, 814           ← very long episodes, no goals scored
- Early episodes are very long (600+ steps) = shaped reward accumulated past the 0.3 threshold, so opponents were updated a few times.

- Now (iteration 925):
- episode_reward_mean: 0.0006  ← essentially 0, callback never fires
- policy_reward_mean/default:    +0.187   ← default is winning!
- policy_reward_mean/opponent_1: -0.421   ← opponent is losing
- policy_reward_mean/opponent_3: -0.491
- Problem: episode_reward_mean is the average across all policies. In self-play one side winning equals the other losing, so the total is close to 0. Even though default is clearly winning (+0.187), the callback condition > 0.3 looks at the total average and never fires. Opponents are frozen again.

#### Fix:
Changed opponent update logic to:
```
default_reward = info["result"].get("policy_reward_mean", {}).get("default", -999)
        if default_reward > 0.2:
```

Resuming training from iteration 900.
- LOG: `soccerstwos-4810130.out`
- `PPO_Soccer_d156b_00000_0_2026-04-12_14-38-37`




#### 1700 vs baseline

```
Progress: |██████████████████████████████████████████████████| 100 / 100 episodes completed
episode_len_mean: 63.79
episode_reward_max: -0.01639997959136963
episode_reward_mean: -0.12717998027801514
episode_reward_min: -0.6763999462127686
episodes_this_eval: 100
policies:
  ceia_baseline_agent:
    blue_team:
      policy_blue_team_draws: 0
      policy_blue_team_losses: 29
      policy_blue_team_reward_max: 1.9808000326156616
      policy_blue_team_reward_mean: -0.3726319968700409
      policy_blue_team_reward_min: -2.0
      policy_blue_team_total_games: 50
      policy_blue_team_win_rate: 0.42
      policy_blue_team_wins: 21
    orange_team:
      policy_orange_team_draws: 0
      policy_orange_team_losses: 33
      policy_orange_team_reward_max: 1.9836000204086304
      policy_orange_team_reward_mean: -0.6750479936599731
      policy_orange_team_reward_min: -2.0
      policy_orange_team_total_games: 50
      policy_orange_team_win_rate: 0.34
      policy_orange_team_wins: 17
    policy_draws: 0
    policy_losses: 62
    policy_reward_max: 1.9836000204086304
    policy_reward_mean: -0.5238400101661682
    policy_reward_min: -2.0
    policy_total_games: 100
    policy_win_rate: 0.38
    policy_wins: 38
  reward_shaping_ppo_agent:
    blue_team:
      policy_blue_team_draws: 0
      policy_blue_team_losses: 17
      policy_blue_team_reward_max: 1.9811999797821045
      policy_blue_team_reward_mean: 0.5624959468841553
      policy_blue_team_reward_min: -2.0
      policy_blue_team_total_games: 50
      policy_blue_team_win_rate: 0.66
      policy_blue_team_wins: 33
    orange_team:
      policy_orange_team_draws: 0
      policy_orange_team_losses: 21
      policy_orange_team_reward_max: 1.9747999906539917
      policy_orange_team_reward_mean: 0.23082399368286133
      policy_orange_team_reward_min: -2.0
      policy_orange_team_total_games: 50
      policy_orange_team_win_rate: 0.58
      policy_orange_team_wins: 29
    policy_draws: 0
    policy_losses: 38
    policy_reward_max: 1.9811999797821045
    policy_reward_mean: 0.3966600000858307
    policy_reward_min: -2.0
    policy_total_games: 100
    policy_win_rate: 0.62
    policy_wins: 62
```

Iteration 1900 is even worse.

The current run at 1900 uses default_reward > 0.2 (policy_reward_mean/default).

Looking at the training CSV data:

iter 901：policy_reward_mean/default ≈ 0.179
iter 902：≈ 0.183
iter 903：≈ 0.180
The problem is: the 0.2 threshold is too low. Every time default_reward slightly exceeds 0.2, opponents are updated, causing them to refresh too frequently. The agent is constantly chasing a target that keeps getting stronger and can never converge.

Suggestion: raise the threshold to 0.3 or 0.4, so the agent genuinely defeats the current opponents before they are updated. Also check how many times "---- Updating opponents!!! ----" appears in the log — if it appears every few dozen iterations, updates are too frequent.



Data is clear:

max is only 0.37, so anything above 0.4 will never trigger
0.3 triggered 46 times (4.3%) — a reasonable frequency
Using 0.3 as the threshold, retraining from checkpoint_1000.

Update train_ray_selfplay.py:


if default_reward > 0.3:   # changed here

restore="./ray_results/PPO_selfplay_rec/PPO_Soccer_36ca0_00000_0_2026-04-07_18-00-41/checkpoint_001000/checkpoint-1000",  # changed here
A 4.3% trigger rate means opponents are updated roughly every 23 iterations, giving the policy enough time to adapt — no more constantly chasing a moving target like the previous 74% rate.


#### My changes
```
if default_reward > 0.3:   # changed here
restore="./ray_results/PPO_selfplay_rec/PPO_Soccer_36ca0_00000_0_2026-04-07_18-00-41/checkpoint_001000/checkpoint-1000",  # changed here
```
new log: `soccerstwos-4814876.out`
`PPO_Soccer_c6e27_00000_0_2026-04-13_00-39-37`

Waiting to see the results in the morning!!



# April 13
- Final training checkpoint: `scratch/soccer-twos/ray_results/PPO_selfplay_rec/PPO_Soccer_c6e27_00000_0_2026-04-13_00-39-37/checkpoint_002296/checkpoint-2296`
- vs baseline comparison:
```
Progress: |██████████████████████████████████████████████████| 100 / 100 episodes completed
episode_len_mean: 74.04
episode_reward_max: -0.0140000581741333
episode_reward_mean: -0.14774800837039948
episode_reward_min: -0.6507999897003174
episodes_this_eval: 100
policies:
  ceia_baseline_agent:
    blue_team:
      policy_blue_team_draws: 0
      policy_blue_team_losses: 24
      policy_blue_team_reward_max: 1.9731999635696411
      policy_blue_team_reward_mean: -0.016247976571321487
      policy_blue_team_reward_min: -2.0
      policy_blue_team_total_games: 50
      policy_blue_team_win_rate: 0.52
      policy_blue_team_wins: 26
    orange_team:
      policy_orange_team_draws: 0
      policy_orange_team_losses: 32
      policy_orange_team_reward_max: 1.981600046157837
      policy_orange_team_reward_mean: -0.6070719361305237
      policy_orange_team_reward_min: -2.0
      policy_orange_team_total_games: 50
      policy_orange_team_win_rate: 0.36
      policy_orange_team_wins: 18
    policy_draws: 0
    policy_losses: 56
    policy_reward_max: 1.981600046157837
    policy_reward_mean: -0.3116599917411804
    policy_reward_min: -2.0
    policy_total_games: 100
    policy_win_rate: 0.44
    policy_wins: 44
  reward_shaping_ppo_agent:
    blue_team:
      policy_blue_team_draws: 0
      policy_blue_team_losses: 18
      policy_blue_team_reward_max: 1.9636000394821167
      policy_blue_team_reward_mean: 0.4777040481567383
      policy_blue_team_reward_min: -2.0
      policy_blue_team_total_games: 50
      policy_blue_team_win_rate: 0.64
      policy_blue_team_wins: 32
    orange_team:
      policy_orange_team_draws: 0
      policy_orange_team_losses: 26
      policy_orange_team_reward_max: 1.9859999418258667
      policy_orange_team_reward_mean: -0.14988000690937042
      policy_orange_team_reward_min: -2.0
      policy_orange_team_total_games: 50
      policy_orange_team_win_rate: 0.48
      policy_orange_team_wins: 24
    policy_draws: 0
    policy_losses: 44
    policy_reward_max: 1.9859999418258667
    policy_reward_mean: 0.16391199827194214
    policy_reward_min: -2.0
    policy_total_games: 100
    policy_win_rate: 0.56
    policy_wins: 56
```


#### Best run is 1800 vs baseline
```
episode_len_mean: 59.31
episode_reward_max: -0.013200044631958008
episode_reward_mean: -0.11812400072813034
episode_reward_min: -0.46480000019073486
episodes_this_eval: 100
policies:
  ceia_baseline_agent:
    blue_team:
      policy_blue_team_draws: 0
      policy_blue_team_losses: 26
      policy_blue_team_reward_max: 1.985200047492981
      policy_blue_team_reward_mean: -0.13680797815322876
      policy_blue_team_reward_min: -2.0
      policy_blue_team_total_games: 50
      policy_blue_team_win_rate: 0.48
      policy_blue_team_wins: 24
    orange_team:
      policy_orange_team_draws: 0
      policy_orange_team_losses: 30
      policy_orange_team_reward_max: 1.9803999662399292
      policy_orange_team_reward_mean: -0.4435279965400696
      policy_orange_team_reward_min: -2.0
      policy_orange_team_total_games: 50
      policy_orange_team_win_rate: 0.4
      policy_orange_team_wins: 20
    policy_draws: 0
    policy_losses: 56
    policy_reward_max: 1.985200047492981
    policy_reward_mean: -0.29016798734664917
    policy_reward_min: -2.0
    policy_total_games: 100
    policy_win_rate: 0.44
    policy_wins: 44
  reward_shaping_ppo_agent:
    blue_team:
      policy_blue_team_draws: 0
      policy_blue_team_losses: 20
      policy_blue_team_reward_max: 1.986799955368042
      policy_blue_team_reward_mean: 0.3308959901332855
      policy_blue_team_reward_min: -2.0
      policy_blue_team_total_games: 50
      policy_blue_team_win_rate: 0.6
      policy_blue_team_wins: 30
    orange_team:
      policy_orange_team_draws: 0
      policy_orange_team_losses: 24
      policy_orange_team_reward_max: 1.972000002861023
      policy_orange_team_reward_mean: 0.013192005455493927
      policy_orange_team_reward_min: -2.0
      policy_orange_team_total_games: 50
      policy_orange_team_win_rate: 0.52
      policy_orange_team_wins: 26
    policy_draws: 0
    policy_losses: 44
    policy_reward_max: 1.986799955368042
    policy_reward_mean: 0.17204400897026062
    policy_reward_min: -2.0
    policy_total_games: 100
    policy_win_rate: 0.56
    policy_wins: 56
```
#### Also maxed out


### New Approach
Starting League training with train_ray_selfplay_league.py.

Core idea: fix ceia_baseline weights as opponent_3, so the default policy has a 15% chance of facing ceia_baseline directly each iteration, forcing it to learn to beat ceia.

Run on the server:


python train_ray_selfplay_league.py
This script will:

Restore from d156b's checkpoint_1000 (72% win rate)
Load ceia_baseline weights into opponent_3 on the first iteration
opponent_3 never updates (fixed target)
opponent_1 and opponent_2 update normally via self-play
Expected effect: policy is forced to learn a strategy against ceia_baseline, not just against clones of itself.

- Updated script: `sbatch scripts/ceia15.batch`
- log: `soccerstwos-4823694.out`
- path: `scratch/soccer-twos/ray_results/PPO_league_ceia/PPO_Soccer_a1374_00000_0_2026-04-13_15-26-12`


#### New Problem:
Issue: policy_reward_mean/default aggregates the rewards of all agents mapped to "default".

This means it includes:

agent 0 (always default, team 0) → +1 on win
agent 1 (default 50% of the time, also on team 0) → +1 on win, positively correlated with agent 0
agent 2/3 (default 50% of the time, on opposing team 1) → -1 on win, cancels out
So roughly half the time default is playing against itself, reward is close to 0, signal is diluted.

Cleaner approach:

Fix teammate (agent 1) to always be default; only let opponents (agent 2, 3) pick randomly:


def policy_mapping_fn(agent_id, *args, **kwargs):
    if agent_id == 0 or agent_id == 1:
        return "default"          # both teammates use default (training team strategy)
    else:
        return np.random.choice(
            ["opponent_1", "opponent_2", "opponent_3"],
            size=1,
            p=[0.50, 0.25, 0.25],
        )[0]
This way policy_reward_mean/default only reflects wins/losses of the default team against the opponent team — a cleaner signal, and the 0.3 threshold becomes meaningful again.

#### Solution:
- Created new `train_PPO_team.py`, `team_ceia15.batch`
- Resuming from checkpoint_1000
- LOG: `soccerstwos-4823988.out`
- PATH: `scratch/soccer-twos/ray_results/PPO_team/PPO_Soccer_0bff0_00000_0_2026-04-13_16-19-17`

#### Result:
1100: 48% is a normal short-term dip, because:

- checkpoint_1000 was trained with "agent 0 = default, agent 1 = random". After switching to "agent 0 and 1 both use default", the opponent policies (opponent_1/2/3) previously had representation from both teams but now all sit in team 1 facing default — effectively the opponents suddenly got stronger.

- Only 100 iterations is far too few to adapt; evaluation timing was too early.

1200:     policy_wins: 41% — still poor

- Waiting to see results at 1300-1500


#### 1300 vs ceia
```
policies:
  ceia_baseline_agent:
    blue_team:
      policy_blue_team_draws: 0
      policy_blue_team_losses: 23
      policy_blue_team_reward_max: 1.9800000190734863
      policy_blue_team_reward_mean: 0.07760000228881836
      policy_blue_team_reward_min: -2.0
      policy_blue_team_total_games: 50
      policy_blue_team_win_rate: 0.54
      policy_blue_team_wins: 27
    orange_team:
      policy_orange_team_draws: 0
      policy_orange_team_losses: 21
      policy_orange_team_reward_max: 1.9808000326156616
      policy_orange_team_reward_mean: 0.2430800199508667
      policy_orange_team_reward_min: -2.0
      policy_orange_team_total_games: 50
      policy_orange_team_win_rate: 0.58
      policy_orange_team_wins: 29
    policy_draws: 0
    policy_losses: 44
    policy_reward_max: 1.9808000326156616
    policy_reward_mean: 0.16034002602100372
    policy_reward_min: -2.0
    policy_total_games: 100
    policy_win_rate: 0.56
    policy_wins: 56
  reward_shaping_ppo_agent:
    blue_team:
      policy_blue_team_draws: 0
      policy_blue_team_losses: 29
      policy_blue_team_reward_max: 1.9687999486923218
      policy_blue_team_reward_mean: -0.3876959979534149
      policy_blue_team_reward_min: -2.0
      policy_blue_team_total_games: 50
      policy_blue_team_win_rate: 0.42
      policy_blue_team_wins: 21
    orange_team:
      policy_orange_team_draws: 0
      policy_orange_team_losses: 27
      policy_orange_team_reward_max: 1.9667999744415283
      policy_orange_team_reward_mean: -0.22411198914051056
      policy_orange_team_reward_min: -2.0
      policy_orange_team_total_games: 50
      policy_orange_team_win_rate: 0.46
      policy_orange_team_wins: 23
    policy_draws: 0
    policy_losses: 56
    policy_reward_max: 1.9687999486923218
    policy_reward_mean: -0.30590400099754333
    policy_reward_min: -2.0
    policy_total_games: 100
    policy_win_rate: 0.44
    policy_wins: 44

```
1500:
```
episode_len_mean: 68.97
episode_reward_max: -0.016000032424926758
episode_reward_mean: -0.1375119835138321
episode_reward_min: -0.5204000473022461
episodes_this_eval: 100
policies:
  ceia_baseline_agent:
    blue_team:
      policy_blue_team_draws: 0
      policy_blue_team_losses: 25
      policy_blue_team_reward_max: 1.9839999675750732
      policy_blue_team_reward_mean: -0.07039999216794968
      policy_blue_team_reward_min: -2.0
      policy_blue_team_total_games: 50
      policy_blue_team_win_rate: 0.5
      policy_blue_team_wins: 25
    orange_team:
      policy_orange_team_draws: 0
      policy_orange_team_losses: 18
      policy_orange_team_reward_max: 1.9775999784469604
      policy_orange_team_reward_mean: 0.472135990858078
      policy_orange_team_reward_min: -2.0
      policy_orange_team_total_games: 50
      policy_orange_team_win_rate: 0.64
      policy_orange_team_wins: 32
    policy_draws: 0
    policy_losses: 43
    policy_reward_max: 1.9839999675750732
    policy_reward_mean: 0.20086799561977386
    policy_reward_min: -2.0
    policy_total_games: 100
    policy_win_rate: 0.57
    policy_wins: 57
  reward_shaping_ppo_agent:
    blue_team:
      policy_blue_team_draws: 0
      policy_blue_team_losses: 32
      policy_blue_team_reward_max: 1.9839999675750732
      policy_blue_team_reward_mean: -0.6100559830665588
      policy_blue_team_reward_min: -2.0
      policy_blue_team_total_games: 50
      policy_blue_team_win_rate: 0.36
      policy_blue_team_wins: 18
    orange_team:
      policy_orange_team_draws: 0
      policy_orange_team_losses: 25
      policy_orange_team_reward_max: 1.9711999893188477
      policy_orange_team_reward_mean: -0.06670399010181427
      policy_orange_team_reward_min: -2.0
      policy_orange_team_total_games: 50
      policy_orange_team_win_rate: 0.5
      policy_orange_team_wins: 25
    policy_draws: 0
    policy_losses: 57
    policy_reward_max: 1.9839999675750732
    policy_reward_mean: -0.33838000893592834
    policy_reward_min: -2.0
    policy_total_games: 100
    policy_win_rate: 0.43
    policy_wins: 43
```
# April 13 8:30PM 
### Found a Problem:
Burst triggers destroy opponent diversity.
Consecutive trigger counts: [5, 4, 4, 3, 3, ...]
Frequent burst triggers cause opponent_1/2 to update to the latest default version almost every time, leaving the opponent pool with essentially only "the last 5 iterations of self" — diversity is lost.

### Changes:
Change 1: Increase ceia exposure ratio (around lines 52-55)
###### Before
p=[0.50, 0.25, 0.25],  # 25% chance vs ceia_baseline each episode
###### After
p=[0.35, 0.25, 0.40],  # 40% vs ceia_baseline, more direct training signal
Change 2: Add cooldown constants (before class LeagueCallback)
###### Add these two lines
OPPONENT_UPDATE_THRESHOLD = 0.3
OPPONENT_UPDATE_COOLDOWN = 20
Change 3: Add cooldown logic in callback (replace entire update section in on_train_result)

### Continuing training from iteration 1600:
- Log: `soccerstwos-4828367.out`
- path: `PPO_Soccer_6dc4e_00000_0_2026-04-13_21-01-12`



# April 14 10:54 AM
- Yesterday's training result: `scratch/soccer-twos/ray_results/PPO_team/PPO_Soccer_6dc4e_00000_0_2026-04-13_21-01-12/checkpoint_002429/checkpoint-2429`
- vs ceia_baseline comparison:
```
episode_len_mean: 61.78
episode_reward_max: -0.014799952507019043
episode_reward_mean: -0.12318400293588638
episode_reward_min: -0.42239999771118164
episodes_this_eval: 100
policies:
  ceia_baseline_agent:
    blue_team:
      policy_blue_team_draws: 0
      policy_blue_team_losses: 34
      policy_blue_team_reward_max: 1.981600046157837
      policy_blue_team_reward_mean: -0.7537040114402771
      policy_blue_team_reward_min: -2.0
      policy_blue_team_total_games: 50
      policy_blue_team_win_rate: 0.32
      policy_blue_team_wins: 16
    orange_team:
      policy_orange_team_draws: 0
      policy_orange_team_losses: 34
      policy_orange_team_reward_max: 1.9844000339508057
      policy_orange_team_reward_mean: -0.7657840251922607
      policy_orange_team_reward_min: -2.0
      policy_orange_team_total_games: 50
      policy_orange_team_win_rate: 0.32
      policy_orange_team_wins: 16
    policy_draws: 0
    policy_losses: 68
    policy_reward_max: 1.9844000339508057
    policy_reward_mean: -0.7597439289093018
    policy_reward_min: -2.0
    policy_total_games: 100
    policy_win_rate: 0.32
    policy_wins: 32
  reward_shaping_ppo_agent:
    blue_team:
      policy_blue_team_draws: 0
      policy_blue_team_losses: 16
      policy_blue_team_reward_max: 1.985200047492981
      policy_blue_team_reward_mean: 0.6431360840797424
      policy_blue_team_reward_min: -2.0
      policy_blue_team_total_games: 50
      policy_blue_team_win_rate: 0.68
      policy_blue_team_wins: 34
    orange_team:
      policy_orange_team_draws: 0
      policy_orange_team_losses: 16
      policy_orange_team_reward_max: 1.9752000570297241
      policy_orange_team_reward_mean: 0.6299840211868286
      policy_orange_team_reward_min: -2.0
      policy_orange_team_total_games: 50
      policy_orange_team_win_rate: 0.68
      policy_orange_team_wins: 34
    policy_draws: 0
    policy_losses: 32
    policy_reward_max: 1.985200047492981
    policy_reward_mean: 0.6365600228309631
    policy_reward_min: -2.0
    policy_total_games: 100
    policy_win_rate: 0.68
    policy_wins: 68
```
checkpoint_2429 achieved 68% win rate!

Full progress summary:

Checkpoint	Overall Win Rate	Blue Team	Orange Team
1000 (original)	72%	—	—
1100-1500	43-48%	Below average	~50%
1600	53%	56%	50%
2429	68%	68%	68%
Two key breakthroughs:

Recovered to near 72% — league training is starting to pay off
Blue/Orange fully symmetric (68%/68%) — policy no longer has directional bias

Training signals are healthy:
iter 1601-1650: opp3(ceia) = -0.108
iter 2401-2429: opp3(ceia) = -0.182  ← consistently negative, ceia losing more and more
Triggers with cooldown: 29/829 (healthy, no bursts)

To push further towards 90%, next steps:

Continue training from checkpoint_2429, adjust two parameters:
- 1. Increase ceia exposure ratio 40% → 50%
p=[0.30, 0.20, 0.50]   # ceia accounts for half of opponents

- 2. Lower threshold slightly to allow more frequent opponent updates
OPPONENT_UPDATE_THRESHOLD = 0.25  # was 0.3

##### Continuing training from checkpoint_2429
- LOG: `soccerstwos-4839548.out`
- PATH: `PPO_Soccer_d3d39_00000_0_2026-04-14_11-08-44`


#### 3000 vs baseline:
```
episode_len_mean: 70.08
episode_reward_max: -0.014799952507019043
episode_reward_mean: -0.13972000777721405
episode_reward_min: -0.6239999532699585
episodes_this_eval: 100
policies:
  ceia_baseline_agent:
    blue_team:
      policy_blue_team_draws: 0
      policy_blue_team_losses: 32
      policy_blue_team_reward_max: 1.9700000286102295
      policy_blue_team_reward_mean: -0.6117119789123535
      policy_blue_team_reward_min: -2.0
      policy_blue_team_total_games: 50
      policy_blue_team_win_rate: 0.36
      policy_blue_team_wins: 18
    orange_team:
      policy_orange_team_draws: 0
      policy_orange_team_losses: 30
      policy_orange_team_reward_max: 1.9780000448226929
      policy_orange_team_reward_mean: -0.44261595606803894
      policy_orange_team_reward_min: -2.0
      policy_orange_team_total_games: 50
      policy_orange_team_win_rate: 0.4
      policy_orange_team_wins: 20
    policy_draws: 0
    policy_losses: 62
    policy_reward_max: 1.9780000448226929
    policy_reward_mean: -0.5271639823913574
    policy_reward_min: -2.0
    policy_total_games: 100
    policy_win_rate: 0.38
    policy_wins: 38
  reward_shaping_ppo_agent:
    blue_team:
      policy_blue_team_draws: 0
      policy_blue_team_losses: 20
      policy_blue_team_reward_max: 1.985200047492981
      policy_blue_team_reward_mean: 0.31301602721214294
      policy_blue_team_reward_min: -2.0
      policy_blue_team_total_games: 50
      policy_blue_team_win_rate: 0.6
      policy_blue_team_wins: 30
    orange_team:
      policy_orange_team_draws: 0
      policy_orange_team_losses: 18
      policy_orange_team_reward_max: 1.9783999919891357
      policy_orange_team_reward_mean: 0.46187201142311096
      policy_orange_team_reward_min: -2.0
      policy_orange_team_total_games: 50
      policy_orange_team_win_rate: 0.64
      policy_orange_team_wins: 32
    policy_draws: 0
    policy_losses: 38
    policy_reward_max: 1.985200047492981
    policy_reward_mean: 0.38744398951530457
    policy_reward_min: -2.0
    policy_total_games: 100
    policy_win_rate: 0.62
    policy_wins: 62
```




# April 14 4:13PM
#### Diagnosis: League is not actually updating
See train_PPO_team.py:
OPPONENT_UPDATE_THRESHOLD = 0.25  # trigger condition
But recent training reward has been consistently 0.10~0.14, never exceeding 0.25. This means:

opponent_1 and opponent_2 have not been updated since restoring from checkpoint-2429!
So the agent is actually playing against:

50% ceia_baseline (effective)
30% opponent_1 = old snapshot from 2429 (stale)
20% opponent_2 = even older snapshot (stale)
The core advantage of League training is completely wasted.

#### Changed threshold=0.08, continuing training from 3100
- LOG: `soccerstwos-4842435.out`
- PATH: `PPO_Soccer_17b64_00000_0_2026-04-14_16-18-26`

# April 14 7:32PM
Added selfmade_random_agent
- Samples random actions at every step
- `python -m soccer_twos.evaluate     -m1 reward_shaping_ppo_agent     -m2 selfmade_random_agent     -e 100`
- Match results:
```
episodes_this_eval: 100
policies:
  reward_shaping_ppo_agent:
    blue_team:
      policy_blue_team_draws: 0
      policy_blue_team_losses: 1
      policy_blue_team_reward_max: 1.9839999675750732
      policy_blue_team_reward_mean: 1.804368019104004
      policy_blue_team_reward_min: -2.0
      policy_blue_team_total_games: 50
      policy_blue_team_win_rate: 0.98
      policy_blue_team_wins: 49
    orange_team:
      policy_orange_team_draws: 0
      policy_orange_team_losses: 0
      policy_orange_team_reward_max: 1.9780000448226929
      policy_orange_team_reward_mean: 1.9036400318145752
      policy_orange_team_reward_min: 1.6411999464035034
      policy_orange_team_total_games: 50
      policy_orange_team_win_rate: 1.0
      policy_orange_team_wins: 50
    policy_draws: 0
    policy_losses: 1
    policy_reward_max: 1.9839999675750732
    policy_reward_mean: 1.8540040254592896
    policy_reward_min: -2.0
    policy_total_games: 100
    policy_win_rate: 0.99
    policy_wins: 99
  selfmade_random_agent:
    blue_team:
      policy_blue_team_draws: 0
      policy_blue_team_losses: 50
      policy_blue_team_reward_max: -2.0
      policy_blue_team_reward_mean: -2.0
      policy_blue_team_reward_min: -2.0
      policy_blue_team_total_games: 50
      policy_blue_team_win_rate: 0.0
      policy_blue_team_wins: 0
    orange_team:
      policy_orange_team_draws: 0
      policy_orange_team_losses: 49
      policy_orange_team_reward_max: 1.8207999467849731
      policy_orange_team_reward_mean: -1.923583984375
      policy_orange_team_reward_min: -2.0
      policy_orange_team_total_games: 50
      policy_orange_team_win_rate: 0.02
      policy_orange_team_wins: 1
    policy_draws: 0
    policy_losses: 99
    policy_reward_max: 1.8207999467849731
    policy_reward_mean: -1.9617919921875
    policy_reward_min: -2.0
    policy_total_games: 100
    policy_win_rate: 0.01
    policy_wins: 1
```


# April 15 2:17PM
- Yesterday's run reached checkpoint 4500
```
episodes_this_eval: 100
policies:
  ceia_baseline_agent:
    blue_team:
      policy_blue_team_draws: 0
      policy_blue_team_losses: 32
      policy_blue_team_reward_max: 1.9759999513626099
      policy_blue_team_reward_mean: -0.6069119572639465
      policy_blue_team_reward_min: -2.0
      policy_blue_team_total_games: 50
      policy_blue_team_win_rate: 0.36
      policy_blue_team_wins: 18
    orange_team:
      policy_orange_team_draws: 0
      policy_orange_team_losses: 34
      policy_orange_team_reward_max: 1.9819999933242798
      policy_orange_team_reward_mean: -0.7531440258026123
      policy_orange_team_reward_min: -2.0
      policy_orange_team_total_games: 50
      policy_orange_team_win_rate: 0.32
      policy_orange_team_wins: 16
    policy_draws: 0
    policy_losses: 66
    policy_reward_max: 1.9819999933242798
    policy_reward_mean: -0.6800280213356018
    policy_reward_min: -2.0
    policy_total_games: 100
    policy_win_rate: 0.34
    policy_wins: 34
  reward_shaping_ppo_agent:
    blue_team:
      policy_blue_team_draws: 0
      policy_blue_team_losses: 16
      policy_blue_team_reward_max: 1.985200047492981
      policy_blue_team_reward_mean: 0.624239981174469
      policy_blue_team_reward_min: -2.0
      policy_blue_team_total_games: 50
      policy_blue_team_win_rate: 0.68
      policy_blue_team_wins: 34
    orange_team:
      policy_orange_team_draws: 0
      policy_orange_team_losses: 18
      policy_orange_team_reward_max: 1.969599962234497
      policy_orange_team_reward_mean: 0.4575920104980469
      policy_orange_team_reward_min: -2.0
      policy_orange_team_total_games: 50
      policy_orange_team_win_rate: 0.64
      policy_orange_team_wins: 32
    policy_draws: 0
    policy_losses: 34
    policy_reward_max: 1.985200047492981
    policy_reward_mean: 0.5409160256385803
    policy_reward_min: -2.0
    policy_total_games: 100
    policy_win_rate: 0.66
    policy_wins: 66
```

#### Analysis:
During training default consistently beats ceia, but evaluated win rate is only 66%. Root contradiction identified:

The real gap: 66% → 90% is not a "quantity" problem, it's a "quality" problem.
Stage	Meaning
50%~70%	Both sides winning and losing, slight advantage
70%~85%	Consistently pressuring, but ceia occasionally pushes back
90%+	Near-dominance required, barely allowing ceia any goals
From training data, ceia's reward fluctuates a lot between -0.2 ~ -0.5, indicating the agent is not stably dominating ceia — ceia can still counter-attack effectively in many games.

Potential bottlenecks
1. Insufficient opponent diversity to push agent to a higher level

In League, opponent_1/2 update every 20 rounds to become the current default, but default itself is only ~16% stronger than ceia. Self-play opponents apply no more "pressure" on the agent than ceia does, so there is no effective upward force.

2. Network [256, 256] may be a ceiling

The current architecture is relatively small and may have hit an expressiveness bottleneck at a certain policy complexity.

3. More training time needed

The trend is correct, but 3000→4500 only improved by 4% — at this rate reaching 90% will take a very long time.

Recommendations
Short term (continue current training):

Keep running, evaluate every 500 checkpoints, check whether the win rate slope is accelerating.
If still below 70% at 5000~6000, the architecture/reward has hit a bottleneck.
Medium term (if win rate stalls):

Consider upgrading network from [256, 256] to [512, 512] or [256, 256, 256] — requires training from scratch.
Or adjust reward shaping to strengthen goal/defence signals.


#### Submitted a job continuing training from 4500
- `soccerstwos-4885770.out`
- `scratch/soccer-twos/ray_results/PPO_team/PPO_Soccer_d115d_00000_0_2026-04-15_14-35-03`

#### Professor's Recommendations
Continue current training to 5500~6000, then evaluate win rate again.

Reasoning:

League only just started working correctly (threshold was fixed at 3100).
Self-play opponent quality is improving; the agent still has room to grow.
Retraining from scratch is expensive and the direction for reward shaping changes is uncertain.
If win rate hasn't reached 75%+ by iteration 6000, then reconsider whether retraining is worthwhile.


##### April 15 5:53PM
- Training interrupted — error in the log
- Resuming from checkpoint 4700
- `soccerstwos-4893401.out`
- `PPO_Soccer_c72dd_00000_0_2026-04-15_17-55-13`

# April 16 7:45PM PDT
- Continuing training from 6100
- LOG: `soccerstwos-4904849.out`
- `PPO_Soccer_d745c_00000_0_2026-04-16_22-33-39`

#### 6100 vs ceia_baseline:

```
episode_len_mean: 65.41
episode_reward_max: -0.017600059509277344
episode_reward_mean: -0.13038800656795502
episode_reward_min: -0.5551999807357788
episodes_this_eval: 100
policies:
  ceia_baseline_agent:
    blue_team:
      policy_blue_team_draws: 0
      policy_blue_team_losses: 34
      policy_blue_team_reward_max: 1.9823999404907227
      policy_blue_team_reward_mean: -0.7600558996200562
      policy_blue_team_reward_min: -2.0
      policy_blue_team_total_games: 50
      policy_blue_team_win_rate: 0.32
      policy_blue_team_wins: 16
    orange_team:
      policy_orange_team_draws: 0
      policy_orange_team_losses: 41
      policy_orange_team_reward_max: 1.9736000299453735
      policy_orange_team_reward_mean: -1.299064040184021
      policy_orange_team_reward_min: -2.0
      policy_orange_team_total_games: 50
      policy_orange_team_win_rate: 0.18
      policy_orange_team_wins: 9
    policy_draws: 0
    policy_losses: 75
    policy_reward_max: 1.9823999404907227
    policy_reward_mean: -1.0295599699020386
    policy_reward_min: -2.0
    policy_total_games: 100
    policy_win_rate: 0.25
    policy_wins: 25
  reward_shaping_ppo_agent:
    blue_team:
      policy_blue_team_draws: 0
      policy_blue_team_losses: 9
      policy_blue_team_reward_max: 1.9767999649047852
      policy_blue_team_reward_mean: 1.172984004020691
      policy_blue_team_reward_min: -2.0
      policy_blue_team_total_games: 50
      policy_blue_team_win_rate: 0.82
      policy_blue_team_wins: 41
    orange_team:
      policy_orange_team_draws: 0
      policy_orange_team_losses: 16
      policy_orange_team_reward_max: 1.9752000570297241
      policy_orange_team_reward_mean: 0.6253600120544434
      policy_orange_team_reward_min: -2.0
      policy_orange_team_total_games: 50
      policy_orange_team_win_rate: 0.68
      policy_orange_team_wins: 34
    policy_draws: 0
    policy_losses: 25
    policy_reward_max: 1.9767999649047852
    policy_reward_mean: 0.8991720080375671
    policy_reward_min: -2.0
    policy_total_games: 100
    policy_win_rate: 0.75
    policy_wins: 75
```

#### Great progress! Trend is accelerating.
Checkpoint	Win Rate vs Ceia
3000	62%
4500	66% (+4%)
6100	75% (+9%)
Improvement rate is increasing; League running normally (59 promotions).

Training data analysis
Average default reward per 100 iterations:


4800-5500: 0.23~0.28  (steady rise)
5500-6100: 0.19~0.23  (slight dip)
Default reward dropped in the last 200 iterations (0.28 → 0.19), but this is actually a good sign — it means opponent_1/2 got stronger and League is normally raising difficulty. Correspondingly, evaluated win rate jumped from 66% to 75%.

Judgement: keep training, no major adjustments needed.
League adaptive mechanism is working; current trajectory prediction:


6100: 75%
~7000: 80%±?
~8000+: approaching 90%?
One signal worth watching: default reward for this 6100 batch averages only 0.19, and ceia's negative value is narrowing (-0.21 vs -0.30 before). May indicate opponent_1/2 are now too strong and pulling default down — needs a few hundred more iterations to observe whether it recovers.

Continue to 7500, then evaluate win rate once more to decide whether adjustment is needed.

# April 17 11:22 AM PDT
- Yesterday's continued training reached checkpoint 7500
```
episodes_this_eval: 100
policies:
  ceia_baseline_agent:
    blue_team:
      policy_blue_team_draws: 0
      policy_blue_team_losses: 38
      policy_blue_team_reward_max: 1.9836000204086304
      policy_blue_team_reward_mean: -1.0592399835586548
      policy_blue_team_reward_min: -2.0
      policy_blue_team_total_games: 50
      policy_blue_team_win_rate: 0.24
      policy_blue_team_wins: 12
    orange_team:
      policy_orange_team_draws: 0
      policy_orange_team_losses: 33
      policy_orange_team_reward_max: 1.9788000583648682
      policy_orange_team_reward_mean: -0.6789119839668274
      policy_orange_team_reward_min: -2.0
      policy_orange_team_total_games: 50
      policy_orange_team_win_rate: 0.34
      policy_orange_team_wins: 17
    policy_draws: 0
    policy_losses: 71
    policy_reward_max: 1.9836000204086304
    policy_reward_mean: -0.8690759539604187
    policy_reward_min: -2.0
    policy_total_games: 100
    policy_win_rate: 0.29
    policy_wins: 29
  reward_shaping_ppo_agent:
    blue_team:
      policy_blue_team_draws: 0
      policy_blue_team_losses: 17
      policy_blue_team_reward_max: 1.9788000583648682
      policy_blue_team_reward_mean: 0.5536959767341614
      policy_blue_team_reward_min: -2.0
      policy_blue_team_total_games: 50
      policy_blue_team_win_rate: 0.66
      policy_blue_team_wins: 33
    orange_team:
      policy_orange_team_draws: 0
      policy_orange_team_losses: 12
      policy_orange_team_reward_max: 1.9859999418258667
      policy_orange_team_reward_mean: 0.9377040266990662
      policy_orange_team_reward_min: -2.0
      policy_orange_team_total_games: 50
      policy_orange_team_win_rate: 0.76
      policy_orange_team_wins: 38
    policy_draws: 0
    policy_losses: 29
    policy_reward_max: 1.9859999418258667
    policy_reward_mean: 0.745699942111969
    policy_reward_min: -2.0
    policy_total_games: 100
    policy_win_rate: 0.71
    policy_wins: 71
```

#### 8100 vs ceia_baseline:
```
episodes_this_eval: 100
policies:
  ceia_baseline_agent:
    blue_team:
      policy_blue_team_draws: 0
      policy_blue_team_losses: 35
      policy_blue_team_reward_max: 1.9859999418258667
      policy_blue_team_reward_mean: -0.8249120116233826
      policy_blue_team_reward_min: -2.0
      policy_blue_team_total_games: 50
      policy_blue_team_win_rate: 0.3
      policy_blue_team_wins: 15
    orange_team:
      policy_orange_team_draws: 0
      policy_orange_team_losses: 42
      policy_orange_team_reward_max: 1.9819999933242798
      policy_orange_team_reward_mean: -1.373184084892273
      policy_orange_team_reward_min: -2.0
      policy_orange_team_total_games: 50
      policy_orange_team_win_rate: 0.16
      policy_orange_team_wins: 8
    policy_draws: 0
    policy_losses: 77
    policy_reward_max: 1.9859999418258667
    policy_reward_mean: -1.0990480184555054
    policy_reward_min: -2.0
    policy_total_games: 100
    policy_win_rate: 0.23
    policy_wins: 23
  reward_shaping_ppo_agent:
    blue_team:
      policy_blue_team_draws: 0
      policy_blue_team_losses: 8
      policy_blue_team_reward_max: 1.9847999811172485
      policy_blue_team_reward_mean: 1.2562799453735352
      policy_blue_team_reward_min: -2.0
      policy_blue_team_total_games: 50
      policy_blue_team_win_rate: 0.84
      policy_blue_team_wins: 42
    orange_team:
      policy_orange_team_draws: 0
      policy_orange_team_losses: 15
      policy_orange_team_reward_max: 1.979599952697754
      policy_orange_team_reward_mean: 0.6731359958648682
      policy_orange_team_reward_min: -2.0
      policy_orange_team_total_games: 50
      policy_orange_team_win_rate: 0.7
      policy_orange_team_wins: 35
    policy_draws: 0
    policy_losses: 23
    policy_reward_max: 1.9847999811172485
    policy_reward_mean: 0.9647080302238464
    policy_reward_min: -2.0
    policy_total_games: 100
    policy_win_rate: 0.77
    policy_wins: 77
```

# April 18 12:00 PM PDT
Yesterday's training reached `checkpoint-8396`, policy_wins: 69. Not good enough.
Making the following changes:
Change 1: Increase ceia ratio (most important)


Before:
p=[0.30, 0.20, 0.50]  # 50% ceia

After:
p=[0.15, 0.15, 0.70]  # 70% ceia
Giving the agent more time to exclusively oppose the fixed baseline, reducing time "held back" by old versions of itself.

Change 2: Add entropy bonus to prevent over-convergence


"model": {
    "vf_share_layers": True,
    "fcnet_hiddens": [256, 256],
    "fcnet_activation": "relu",
},
"entropy_coeff": 0.01,  # add this line
Entropy fluctuating between 0.08-0.24 indicates the policy is already somewhat deterministic; adding an entropy bonus keeps the agent exploring.

#### Continuing training from `checkpoint-8396`

- Log: soccerstwos-4955097.out
- Path: PPO_team/PPO_Soccer_ef501_00000_0_2026-04-18_15-08-08
- Timed out, the above two entries deleted
- New log: soccerstwos-4955667.out
- path: PPO_Soccer_05196_00000_0_2026-04-18_15-23-03


# April 21 1:10PM
The problem is clear. In this new run, 781 out of 1442 iterations (54%) had default_reward > 0.25 — the arms race is still severe; threshold 0.25 is too easy to trigger in the early phase (when reward reached 0.55).

The more fundamental problem: self-play itself is the culprit.

Analysis:

85% ceia + 15% self-play, but each time an opponent is promoted, the self-play component has R≈0 (both sides equal).
default_reward ≈ 0.85 × R(vs ceia) + 0 → early 0.37, late 0.23, showing vs-ceia performance is genuinely degrading.
Self-play promoted opponents are close in strength to default, causing gradient updates to start optimising a "counter self" strategy that drifts away from ceia's playstyle.
Most direct fix: remove self-play entirely and go 100% vs ceia.

#### Created train_PPO_ceia_fixed.py
Continuing training from checkpoint-12600


## Created train_PPO_large.py with 512×512 network
Comparison with existing training:

Parameter	train_PPO_team.py (256)	train_PPO_large.py (512)
Network	[256, 256], shared VF	[512, 512], separate VF
Opponent	100% ceia (just changed)	100% ceia
Entropy	fixed 0.01	0.01 → 0 (annealing)
batch size	4000	8000
sgd iter	10 (default)	15
restore	checkpoint_012600	train from scratch
Output dir	ray_results/PPO_team	ray_results/PPO_large
Both runs can proceed simultaneously without interference. The 512 version will take longer to surpass the current win rate of 256 (since it starts from scratch), but has a higher ceiling.


### Learned some new tricks — hyperparameter tuning
RLlib PPO defaults vs Unity Soccer official vs recommended:

Parameter	RLlib default	Unity Soccer official	Recommended
lr	5e-5	3e-4	3e-4 + linear decay
clip_param	0.3	0.2	0.2
lambda (GAE)	1.0	0.95	0.95
entropy_coeff	0.0	0.005	0.005 (fixed) or schedule
num_sgd_iter	30	3	10
vf_loss_coeff	1.0	—	0.5 (when sharing layers)
clip_param and lambda are the two most critical:

clip_param=0.3 is too large — update steps are too aggressive and can destroy good policies.
lambda=1.0 (default) equals Monte Carlo return with high variance; GAE at 0.95 balances bias-variance.


See git for the exact changes. 


# April 22 12:23PM
#### $[256,256]$ version
- PPO_team/PPO_Soccer_ec722_00000_0_2026-04-22_00-01-22/checkpoint_015000/checkpoint-15000
```
Progress: |██████████████████████████████████████████████████| 100 / 100 episodes completed
episode_len_mean: 55.37
episode_reward_max: -0.0196000337600708
episode_reward_mean: -0.11042799800634384
episode_reward_min: -0.4235999584197998
episodes_this_eval: 100
policies:
  ceia_baseline_agent:
    blue_team:
      policy_blue_team_draws: 0
      policy_blue_team_losses: 41
      policy_blue_team_reward_max: 1.9772000312805176
      policy_blue_team_reward_mean: -1.2900558710098267
      policy_blue_team_reward_min: -2.0
      policy_blue_team_total_games: 50
      policy_blue_team_win_rate: 0.18
      policy_blue_team_wins: 9
    orange_team:
      policy_orange_team_draws: 0
      policy_orange_team_losses: 35
      policy_orange_team_reward_max: 1.9803999662399292
      policy_orange_team_reward_mean: -0.8217759728431702
      policy_orange_team_reward_min: -2.0
      policy_orange_team_total_games: 50
      policy_orange_team_win_rate: 0.3
      policy_orange_team_wins: 15
    policy_draws: 0
    policy_losses: 76
    policy_reward_max: 1.9803999662399292
    policy_reward_mean: -1.0559159517288208
    policy_reward_min: -2.0
    policy_total_games: 100
    policy_win_rate: 0.24
    policy_wins: 24
  reward_shaping_ppo_agent:
    blue_team:
      policy_blue_team_draws: 0
      policy_blue_team_losses: 15
      policy_blue_team_reward_max: 1.9764000177383423
      policy_blue_team_reward_mean: 0.7082480788230896
      policy_blue_team_reward_min: -2.0
      policy_blue_team_total_games: 50
      policy_blue_team_win_rate: 0.7
      policy_blue_team_wins: 35
    orange_team:
      policy_orange_team_draws: 0
      policy_orange_team_losses: 9
      policy_orange_team_reward_max: 1.9752000570297241
      policy_orange_team_reward_mean: 1.1827279329299927
      policy_orange_team_reward_min: -2.0
      policy_orange_team_total_games: 50
      policy_orange_team_win_rate: 0.82
      policy_orange_team_wins: 41
    policy_draws: 0
    policy_losses: 24
    policy_reward_max: 1.9764000177383423
    policy_reward_mean: 0.9454880356788635
    policy_reward_min: -2.0
    policy_total_games: 100
    policy_win_rate: 0.76
    policy_wins: 76
```

#### $[512,512]$ version
- scratch/soccer-twos/ray_results/PPO_large/PPO_Soccer_c6af9_00000_0_2026-04-21_23-46-00/checkpoint_001500/checkpoint-1500
```
Progress: |██████████████████████████████████████████████████| 100 / 100 episodes completed
episode_len_mean: 43.37
episode_reward_max: -0.02199995517730713
episode_reward_mean: -0.08634399622678757
episode_reward_min: -0.25199997425079346
episodes_this_eval: 100
policies:
  ceia_baseline_agent:
    blue_team:
      policy_blue_team_draws: 0
      policy_blue_team_losses: 37
      policy_blue_team_reward_max: 1.975600004196167
      policy_blue_team_reward_mean: -0.9819920063018799
      policy_blue_team_reward_min: -2.0
      policy_blue_team_total_games: 50
      policy_blue_team_win_rate: 0.26
      policy_blue_team_wins: 13
    orange_team:
      policy_orange_team_draws: 0
      policy_orange_team_losses: 42
      policy_orange_team_reward_max: 1.9775999784469604
      policy_orange_team_reward_mean: -1.371832013130188
      policy_orange_team_reward_min: -2.0
      policy_orange_team_total_games: 50
      policy_orange_team_win_rate: 0.16
      policy_orange_team_wins: 8
    policy_draws: 0
    policy_losses: 79
    policy_reward_max: 1.9775999784469604
    policy_reward_mean: -1.1769120693206787
    policy_reward_min: -2.0
    policy_total_games: 100
    policy_win_rate: 0.21
    policy_wins: 21
  reward_shaping_ppo_agent:
    blue_team:
      policy_blue_team_draws: 0
      policy_blue_team_losses: 8
      policy_blue_team_reward_max: 1.9780000448226929
      policy_blue_team_reward_mean: 1.2899121046066284
      policy_blue_team_reward_min: -2.0
      policy_blue_team_total_games: 50
      policy_blue_team_win_rate: 0.84
      policy_blue_team_wins: 42
    orange_team:
      policy_orange_team_draws: 0
      policy_orange_team_losses: 13
      policy_orange_team_reward_max: 1.9780000448226929
      policy_orange_team_reward_mean: 0.8912240862846375
      policy_orange_team_reward_min: -2.0
      policy_orange_team_total_games: 50
      policy_orange_team_win_rate: 0.74
      policy_orange_team_wins: 37
    policy_draws: 0
    policy_losses: 21
    policy_reward_max: 1.9780000448226929
    policy_reward_mean: 1.0905680656433105
    policy_reward_min: -2.0
    policy_total_games: 100
    policy_win_rate: 0.79
    policy_wins: 79
```

Overall comparison:

Version	Current iter	Win Rate	Training reward	Trend
256 (PPO_team)	~15000	76%	0.37→0.67	↑ still rising fast
512 (PPO_large)	~1500	79%	~1.09 (stable)	→ converged early
Previous best was only 64%; both now exceed that. Hyperparameter tuning (lr=3e-4, clip=0.2, lambda=0.95) is clearly very effective.

Per-version analysis:

256: Training reward rose from 0.37 to 0.67 in this new run; slope is still increasing (iter 14900-15000 avg=0.669) — no sign of convergence, will keep rising.

512: Reached 79% in only 1500 iterations, but reward has been stable at 1.09 since iter 500. Need to determine whether it has genuinely converged or still has room. Surpassing 256's 15000-iter result in so few iterations confirms larger networks are indeed more efficient.

Conclusion: no changes needed for either; keep running.

Next evaluation: wait for 256 to reach ~iter 16000-17000 and check whether reward is still rising; wait for 512 to reach ~iter 3000-4000 to judge whether it is truly stuck or slowly improving.

# April 23 2:00 PM
`python -m soccer_twos.evaluate     -m1 reward_shaping_ppo_team_256     -m2 41_AGENT     -e 100`
Match results:
episodes_this_eval: 100
policies:
  41_AGENT:
    blue_team:
      policy_blue_team_wins: 28
    orange_team:
      policy_orange_team_wins: 35
    policy_wins: 63
  reward_shaping_ppo_team_256:
    blue_team:
      policy_blue_team_wins: 15
    orange_team:
      policy_orange_team_wins: 22
    policy_wins: 37


`python -m soccer_twos.evaluate     -m1 reward_shaping_ppo_team_512     -m2 41_AGENT     -e 100`
Match results:
episodes_this_eval: 100
policies:
  41_AGENT:
    blue_team:
      policy_blue_team_wins: 30
    orange_team:
      policy_orange_team_wins: 41
    policy_wins: 71
  reward_shaping_ppo_team_512:
    blue_team:
      policy_blue_team_wins: 9
    orange_team:
      policy_orange_team_wins: 20
    policy_wins: 29

